In [ ]:
# ============================================================
# Cell 0: 环境自检（本章纯 CPU 即可，无需 GPU）
# ============================================================
# 本章把第 6-8 章的零件拼成一个完整的 Decoder-only 模型，再做一个深层 Pre-LN /
# Post-LN 的训练稳定性对照实验——全程在维度 ≤ 128 的小张量上跑，CPU 一两分钟即可，
# 不强制 GPU。最后读 Qwen3-8B 的 config 只下载几 KB 的 json。
import sys, platform
import torch

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "（本章用不到，CPU 即可）")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须是 cell 第一行，把 pip 安装日志藏起来。
# torch / matplotlib: 纯 CPU 跑小模型 + 画 loss 曲线，Colab 默认已装，故意【不】加 -U。
# transformers:       第 7.6 节用 AutoConfig 读 Qwen3-8B 的 config，Qwen3 要求 >=4.51。
!pip install -q -U "transformers>=4.51"

In [ ]:
# ============================================================
# Cell 2: 复现第 6-8 章的零件，并拼出一个 DecoderBlock
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class RMSNorm(nn.Module):
    """只按均方根缩放、不减均值的归一化（第 8 章）。x: [..., d] -> [..., d]。"""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))   # 每个特征一个可学习增益 γ
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()  # 1/RMS，形状 [..., 1]
        return x * rms * self.weight                                 # 广播回 [..., d]

def build_rope_cache(seq_len, head_dim, base=10000.0):
    """预先算好每个位置 × 每个频率的 cos/sin（第 4 章 RoPE）。返回两个 [seq_len, head_dim]。"""
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))  # [head_dim/2]
    pos = torch.arange(seq_len).float()                                          # [L]
    ang = torch.outer(pos, inv_freq)                                             # [L, head_dim/2]
    cos = torch.cat([ang.cos(), ang.cos()], dim=-1)                              # [L, head_dim]
    sin = torch.cat([ang.sin(), ang.sin()], dim=-1)                              # [L, head_dim]
    return cos, sin

def apply_rope(x, cos, sin):
    """把每个 head 的向量按位置旋转。x: [B, H, L, hd]；cos/sin: [L, hd]。"""
    cos, sin = cos[None, None], sin[None, None]              # [1,1,L,hd] 便于广播
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2:]              # 前一半 / 后一半配成旋转对
    rotated = torch.cat([-x2, x1], dim=-1)                  # 旋转 90° 的"虚部"
    return x * cos + rotated * sin

class CausalGQA(nn.Module):
    """因果分组查询注意力（第 6、7 章）：H 个 Q 头、G 个 KV 头，含 RoPE 与上三角掩码。"""
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)     # 现代 LLM 投影不带 bias
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)  # GQA：KV 头更少 -> 更窄
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(n_heads * self.head_dim, d_model, bias=False)
    def forward(self, x, cos, sin):
        B, L, _ = x.shape
        H, G, hd = self.n_heads, self.n_kv_heads, self.head_dim
        q = self.q_proj(x).view(B, L, H, hd).transpose(1, 2)   # [B, H, L, hd]
        k = self.k_proj(x).view(B, L, G, hd).transpose(1, 2)   # [B, G, L, hd]
        v = self.v_proj(x).view(B, L, G, hd).transpose(1, 2)   # [B, G, L, hd]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        rep = H // G
        k = k.repeat_interleave(rep, dim=1)                    # 复制凑齐 H 份 -> [B, H, L, hd]
        v = v.repeat_interleave(rep, dim=1)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(hd)     # [B, H, L, L]
        mask = torch.triu(torch.full((L, L), float("-inf")), diagonal=1)  # 上三角 -inf
        attn = (scores + mask).softmax(dim=-1)
        out = attn @ v                                         # [B, H, L, hd]
        out = out.transpose(1, 2).contiguous().view(B, L, H * hd)  # 合头 -> [B, L, d_model]
        return self.o_proj(out)

class SwiGLU(nn.Module):
    """SwiGLU 前馈网络（第 8 章）：SiLU(gate) ⊙ up，再降维。三个无 bias 线性层。"""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class DecoderBlock(nn.Module):
    """一个 Transformer 层：注意力子层 + FFN 子层，各裹 RMSNorm 与残差。
    pre_ln=True 是现代默认（Pre-LN）；False 切到原版 Post-LN，仅用于第 7.5 节对照实验。"""
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, pre_ln=True):
        super().__init__()
        self.pre_ln = pre_ln
        self.attn  = CausalGQA(d_model, n_heads, n_kv_heads)
        self.ffn   = SwiGLU(d_model, d_ff)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
    def forward(self, x, cos, sin):
        if self.pre_ln:                                       # x + Sublayer(Norm(x))
            x = x + self.attn(self.norm1(x), cos, sin)
            x = x + self.ffn(self.norm2(x))
        else:                                                 # Norm(x + Sublayer(x))
            x = self.norm1(x + self.attn(x, cos, sin))
            x = self.norm2(x + self.ffn(x))
        return x

In [ ]:
# ============================================================
# Cell 3: 把零件拼成完整的 Decoder-only 语言模型
# ============================================================
class MiniDecoderLM(nn.Module):
    """把零件拼成完整的 Decoder-only 语言模型：embedding -> N×block -> final norm -> lm_head。"""
    def __init__(self, vocab, d_model=64, n_layers=4, n_heads=4, n_kv_heads=2,
                 d_ff=None, max_len=64, pre_ln=True):
        super().__init__()
        if d_ff is None:
            d_ff = (int(8 / 3 * d_model) + 7) // 8 * 8        # SwiGLU 对齐参数（约 8/3·d），取整到 8 的倍数
        self.embed = nn.Embedding(vocab, d_model)            # [V, d] 查表
        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, n_kv_heads, d_ff, pre_ln) for _ in range(n_layers)
        ])
        self.final_norm = RMSNorm(d_model)                   # 末层归一化（不属于任何 block）
        self.lm_head = nn.Linear(d_model, vocab, bias=False) # [d, V] 投回词表
        cos, sin = build_rope_cache(max_len, d_model // n_heads)
        self.register_buffer("cos", cos)                     # RoPE 表不是参数，注册成 buffer
        self.register_buffer("sin", sin)
    def forward(self, ids):
        B, L = ids.shape
        x = self.embed(ids)                                  # [B, L, d]
        cos, sin = self.cos[:L], self.sin[:L]
        for blk in self.blocks:
            x = blk(x, cos, sin)                             # 每层形状守恒 [B, L, d]
        x = self.final_norm(x)
        return self.lm_head(x)                               # [B, L, V]

In [ ]:
# ============================================================
# Cell 4: 前向一遍——逐层形状守恒 + 参数账
# ============================================================
torch.manual_seed(0)
vocab, B, L = 100, 2, 8
model = MiniDecoderLM(vocab, d_model=64, n_layers=4, n_heads=4, n_kv_heads=2)
ids = torch.randint(0, vocab, (B, L))

# 手动走一遍，确认每个 block 都"形状守恒"
x = model.embed(ids)
print("input_ids       :", tuple(ids.shape), ids.dtype)
print("after embed     :", tuple(x.shape))
cos, sin = model.cos[:L], model.sin[:L]
for i, blk in enumerate(model.blocks):
    x = blk(x, cos, sin)
    print(f"after block[{i}]   :", tuple(x.shape))
x = model.final_norm(x)
print("after final_norm:", tuple(x.shape))
logits = model.lm_head(x)
print("logits          :", tuple(logits.shape), "  <- [B, L, vocab]")

def count(m):
    return sum(p.numel() for p in m.parameters())
print("-" * 40)
print("embed   参数:", count(model.embed))
print("1 block 参数:", count(model.blocks[0]),
      "(attn", count(model.blocks[0].attn), "+ ffn", count(model.blocks[0].ffn), ")")
print("lm_head 参数:", count(model.lm_head))
print("整模型  参数:", count(model))

In [ ]:
# ============================================================
# Cell 5: Pre-LN vs Post-LN —— 24 层深模型的训练稳定性对照（CPU 约一两分钟）
# ============================================================
import matplotlib.pyplot as plt

def train_curve(pre_ln, warmup=0, steps=150, n_layers=24, d_model=128,
                n_heads=8, n_kv_heads=2, L=24, lr=2e-3, vocab=64, seed=0):
    """在一个深(24 层)模型上训一个玩具任务(把序列循环右移 1 位)，返回 loss 曲线。
    warmup>0 时，前 warmup 步把 lr 从 0 线性升到 lr（其余步保持 lr）。"""
    torch.manual_seed(seed)
    m = MiniDecoderLM(vocab, d_model=d_model, n_layers=n_layers,
                      n_heads=n_heads, n_kv_heads=n_kv_heads, max_len=L, pre_ln=pre_ln)
    opt = torch.optim.AdamW(m.parameters(), lr=lr)
    losses = []
    for s in range(steps):
        cur_lr = lr * min(1.0, (s + 1) / warmup) if warmup > 0 else lr   # 线性 warmup
        for g in opt.param_groups:
            g["lr"] = cur_lr
        ids = torch.randint(0, vocab, (8, L))
        tgt = torch.roll(ids, shifts=1, dims=1)             # 确定性映射，模型学得会
        loss = F.cross_entropy(m(ids).reshape(-1, vocab), tgt.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

# 三条曲线（同样 24 层、同样 lr）。CPU 上约一两分钟。
pre     = train_curve(pre_ln=True,  warmup=0)    # Pre-LN，不加 warmup
post    = train_curve(pre_ln=False, warmup=0)    # Post-LN，不加 warmup
post_wu = train_curve(pre_ln=False, warmup=30)   # Post-LN，加 30 步 warmup
print(f"末步 loss —— Pre-LN(无warmup): {pre[-1]:.2f}   "
      f"Post-LN(无warmup): {post[-1]:.2f}   Post-LN(+warmup): {post_wu[-1]:.2f}")

plt.figure(figsize=(7.5, 4.2))
plt.plot(pre,     label="Pre-LN  (no warmup)")
plt.plot(post,    label="Post-LN (no warmup)")
plt.plot(post_wu, label="Post-LN (+ warmup 30)")
plt.xlabel("training step"); plt.ylabel("loss")
plt.title("Pre-LN vs Post-LN: training a 24-layer model (no warmup)")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell 6: 读真实 Qwen3-8B config，按它的尺寸重算参数账（对上全景图）
# ============================================================
# AutoConfig 只下载几 KB 的 config.json，不下载几个 GB 的权重，CPU 秒级完成。
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained("Qwen/Qwen3-8B")
V, d = cfg.vocab_size, cfg.hidden_size
H, G = cfg.num_attention_heads, cfg.num_key_value_heads
hd = getattr(cfg, "head_dim", d // H)   # Qwen3 显式给出 head_dim；它不一定等于 d/H（如 Qwen3-4B 就不等）
d_ff, n_layers = cfg.intermediate_size, cfg.num_hidden_layers
tie = cfg.tie_word_embeddings

emb  = V * d                          # token embedding
q    = d * (H * hd)                   # q_proj
kv   = 2 * d * (G * hd)               # k_proj + v_proj（GQA：KV 头少，所以更窄）
o    = (H * hd) * d                   # o_proj
attn = q + kv + o
ffn  = 3 * d * d_ff                   # SwiGLU 三个矩阵
per_layer = attn + ffn
blocks = per_layer * n_layers
head = 0 if tie else V * d            # 8B 不绑定权重，lm_head 单独一份
total = emb + blocks + head

M = lambda x: f"{x / 1e6:6.0f} M"
print(f"vocab={V}  d_model={d}  layers={n_layers}  H={H}  G={G}  head_dim={hd}  d_ff={d_ff}  tie={tie}")
print("-" * 52)
print("token embedding   :", M(emb))
print("  attn  / layer   :", M(attn), f"(q{M(q)} + kv{M(kv)} + o{M(o)} )")
print("  ffn   / layer   :", M(ffn), " <- 约占一层的 3/4")
print(f"  block ×{n_layers}      : {blocks / 1e9:.2f} B")
print("lm_head           :", M(head), "(tied)" if tie else "(untied)")
print("-" * 52)
print(f"TOTAL             : {total / 1e9:.2f} B   <- 这就是 Qwen3-8B 的 ~8B")